# 🤖 Искусственный интеллект на Python
## Практическое занятие №2 — Консольный ИИ-бот

**Для кого:** 5–8 класс  
**Продолжительность:** около 80 минут  
**Формат:** почти полностью практика

На прошлом занятии наша нейросеть работала с числами.

Сегодня научим её работать с **текстом** и создадим собственного консольного чат-бота.

В конце занятия бот сможет понимать фразы вроде:

```text
привет
что такое python
расскажи про искусственный интеллект
спасибо
пока
```

Главная идея:

```text
ТЕКСТ
  ↓
ПРЕВРАЩАЕМ В ЧИСЛА
  ↓
НЕЙРОСЕТЬ
  ↓
ОПРЕДЕЛЯЕМ ТИП СООБЩЕНИЯ
  ↓
ОТВЕТ БОТА
```

# 0. Подготовка

Сегодня понадобятся:

- `scikit-learn` — машинное обучение;
- `pandas` — удобная работа с таблицами.

Если библиотек нет, раскомментируйте первую строку.

In [ ]:
# !pip install scikit-learn pandas

import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.neural_network import MLPClassifier

print("Библиотеки загружены 🚀")

# Часть 1. Создаём Dataset

**Dataset** — это набор примеров, на которых будет учиться ИИ.

У каждого примера будут:

- `text` — сообщение пользователя;
- `intent` — что пользователь хотел сказать.

Например:

```text
"привет" → hello
"что такое python" → python
"спасибо" → thanks
```

`intent` можно перевести как **намерение**.

In [ ]:
data = {
    "text": [
        # Приветствия
        "привет",
        "здравствуй",
        "добрый день",
        "привет бот",
        "доброе утро",
        "здарова",

        # Python
        "что такое python",
        "расскажи про python",
        "что за язык python",
        "для чего нужен python",
        "python это что",
        "зачем изучать python",

        # Искусственный интеллект
        "что такое искусственный интеллект",
        "что такое ии",
        "расскажи про нейросети",
        "как работает ии",
        "что умеет искусственный интеллект",
        "что такое нейронная сеть",

        # Благодарность
        "спасибо",
        "большое спасибо",
        "спасибо тебе",
        "благодарю",
        "круто спасибо",
        "спасибо бот",

        # Прощание
        "пока",
        "до свидания",
        "увидимся",
        "до встречи",
        "мне пора",
        "прощай"
    ],

    "intent": [
        "hello", "hello", "hello", "hello", "hello", "hello",
        "python", "python", "python", "python", "python", "python",
        "ai", "ai", "ai", "ai", "ai", "ai",
        "thanks", "thanks", "thanks", "thanks", "thanks", "thanks",
        "bye", "bye", "bye", "bye", "bye", "bye"
    ]
}

print("Количество примеров:", len(data["text"]))

## Посмотрим на Dataset как на таблицу

In [ ]:
df = pd.DataFrame(data)
df

Обратите внимание:

```text
text                  intent
--------------------------------
привет                hello
что такое python      python
что такое ии          ai
спасибо               thanks
пока                  bye
```

Мы сами сказали ИИ, какие фразы к какой категории относятся.

Это называется **обучение с учителем**.

# Часть 2. Компьютер не понимает слова как человек

Нейросеть работает с числами.

Но у нас сейчас текст:

```python
"что такое python"
```

Нужно каким-то способом превратить слова в числа.

Для этого используем:

```python
TfidfVectorizer
```

Он анализирует слова и создаёт числовое представление текста.

In [ ]:
vectorizer = TfidfVectorizer()

X = vectorizer.fit_transform(data["text"])

print("Текст преобразован в числа!")
print("Количество сообщений:", X.shape[0])
print("Количество найденных слов:", X.shape[1])

## Какие слова нашёл TF-IDF?

In [ ]:
words = vectorizer.get_feature_names_out()

print(words)

Попробуйте найти среди слов:

```text
привет
python
нейросети
спасибо
пока
```

Каждое слово теперь является одним из признаков для модели.

# Часть 3. Посмотрим, как выглядит предложение в виде чисел

Возьмём фразу:

```text
привет бот
```

и попросим TF-IDF преобразовать её.

In [ ]:
example = vectorizer.transform(["привет бот"])

print(example.toarray())

Получился большой набор чисел.

Большинство значений — `0`.

Ненулевые числа появляются около слов, которые встречаются в предложении.

То есть примерно:

```text
"привет бот"
      ↓
[0, 0, 0.7, 0, 0, 0.5, ...]
```

Нейросеть уже может работать с такими данными.

## Найдём ненулевые слова

Так будет намного понятнее.

In [ ]:
example = vectorizer.transform(["привет бот"]).toarray()[0]

for word, value in zip(words, example):
    if value > 0:
        print(word, "→", round(value, 3))

# Часть 4. Категории тоже превращаем в числа

Наши ответы сейчас выглядят так:

```text
hello
python
ai
thanks
bye
```

Но нейросети удобнее работать с числами.

Используем:

```python
LabelEncoder
```

In [ ]:
encoder = LabelEncoder()

y = encoder.fit_transform(data["intent"])

print("Категории:")
print(encoder.classes_)

print("\nПервые числовые ответы:")
print(y[:15])

Например, категории могут стать такими:

```text
ai      → 0
bye     → 1
hello   → 2
python  → 3
thanks  → 4
```

Точные номера назначает программа автоматически.

# Часть 5. Создаём нейронную сеть

Используем тот же `MLPClassifier`, что и на прошлом занятии.

Теперь его задача сложнее:

не два ответа `0/1`, а сразу несколько категорий.

In [ ]:
model = MLPClassifier(
    hidden_layer_sizes=(12,),
    solver="lbfgs",
    max_iter=3000,
    random_state=42
)

print("Нейронная сеть создана!")

# Часть 6. Обучаем модель

Команда уже знакомая:

```python
fit()
```

In [ ]:
model.fit(X, y)

print("Обучение завершено 🧠")

# Часть 7. Проверяем отдельную фразу

Попробуем:

```text
привет компьютер
```

Сначала превращаем текст в числа, потом передаём модели.

In [ ]:
message = "привет компьютер"

message_vector = vectorizer.transform([message])

prediction = model.predict(message_vector)

intent = encoder.inverse_transform(prediction)[0]

print("Сообщение:", message)
print("Категория:", intent)

Попробуйте заменить сообщение на:

```text
расскажи про python
что такое нейросеть
спасибо большое
до встречи
```

In [ ]:
tests = [
    "расскажи про python",
    "что такое нейросеть",
    "спасибо большое",
    "до встречи"
]

for message in tests:
    message_vector = vectorizer.transform([message])
    prediction = model.predict(message_vector)
    intent = encoder.inverse_transform(prediction)[0]

    print(message, "→", intent)

# Часть 8. Создаём ответы бота

Пока модель умеет определять только категорию:

```text
hello
python
ai
thanks
bye
```

Теперь каждой категории дадим нормальный ответ.

In [ ]:
responses = {
    "hello": "Привет! 👋 Чем могу помочь?",
    "python": "Python — популярный язык программирования. Он подходит для обучения, сайтов, игр, анализа данных и ИИ. 🐍",
    "ai": "Искусственный интеллект — это программы, которые могут учиться на данных и делать предсказания. 🤖",
    "thanks": "Пожалуйста! 😊",
    "bye": "Пока! До встречи! 👋"
}

print(responses["hello"])

# Часть 9. Собираем всё в одну функцию

Создадим функцию:

```python
bot_answer(message)
```

Она будет:

1. принимать текст;
2. превращать его в числа;
3. передавать нейросети;
4. узнавать категорию;
5. возвращать ответ.

In [ ]:
def bot_answer(message):
    message_vector = vectorizer.transform([message])

    prediction = model.predict(message_vector)

    intent = encoder.inverse_transform(prediction)[0]

    return responses[intent]

In [ ]:
print(bot_answer("привет"))
print(bot_answer("что такое python"))
print(bot_answer("объясни нейросети"))
print(bot_answer("спасибо"))

## Теперь попробуйте сами

Придумайте минимум **5 разных сообщений**.

Попробуйте:

- написать правильно;
- сделать небольшую ошибку;
- поменять порядок слов;
- задать странный вопрос.

Например:

```python
bot_answer("python расскажи мне")
bot_answer("привееет")
bot_answer("сколько весит слон")
```

In [ ]:
messages = [
    "python расскажи мне",
    "привееет",
    "сколько весит слон"
]

for message in messages:
    print("Ты:", message)
    print("Бот:", bot_answer(message))
    print()

# Часть 10. Уверенность модели

Есть проблема.

Если написать:

```text
сколько весит слон
```

модель всё равно **обязана выбрать одну из известных категорий**.

Даже если она ничего не знает о слонах.

Поэтому посмотрим на уверенность модели через:

```python
predict_proba()
```

In [ ]:
message = "сколько весит слон"

message_vector = vectorizer.transform([message])

probabilities = model.predict_proba(message_vector)[0]

for intent_name, probability in zip(encoder.classes_, probabilities):
    print(
        intent_name,
        "→",
        f"{probability * 100:.1f}%"
    )

## Найдём самый вероятный ответ

In [ ]:
confidence = max(probabilities)

print("Максимальная уверенность:")
print(f"{confidence * 100:.1f}%")

# Часть 11. Улучшаем функцию бота

Если модель не уверена, бот будет честно говорить:

```text
Я пока не знаю, как на это ответить.
```

Это лучше, чем случайно выдавать неправильный ответ.

In [ ]:
def bot_answer(message):
    message_vector = vectorizer.transform([message])

    probabilities = model.predict_proba(message_vector)[0]

    confidence = max(probabilities)

    if confidence < 0.55:
        return "Я пока не знаю, как на это ответить 😕"

    prediction = model.predict(message_vector)

    intent = encoder.inverse_transform(prediction)[0]

    return responses[intent]

In [ ]:
tests = [
    "привет",
    "что такое python",
    "спасибо",
    "сколько весит слон",
    "какая сегодня погода"
]

for message in tests:
    print("Ты:", message)
    print("Бот:", bot_answer(message))
    print()

# Часть 12. Делаем настоящего консольного чат-бота

Теперь добавим бесконечный цикл.

Бот будет работать до тех пор, пока пользователь не напишет:

```text
выход
```

In [ ]:
print("🤖 ИИ-бот запущен!")
print("Напиши 'выход', чтобы закончить разговор.")

while True:
    message = input("Ты: ")

    if message.lower() == "выход":
        print("Бот: Пока! 👋")
        break

    answer = bot_answer(message)

    print("Бот:", answer)

# Часть 13. Попробуйте «сломать» своего ИИ 😈

Теперь самое интересное.

Попробуйте задать боту вопросы, которых нет в Dataset.

Например:

```text
как тебя зовут?
ты робот?
сколько тебе лет?
какой твой любимый цвет?
кто создал python?
расскажи про minecraft
2 + 2
```

Запишите минимум **3 сообщения**, на которые бот ответил неправильно или сказал, что не знает ответа.

Это нормально.

ИИ знает только то, чему мы его учили.

# Часть 14. Добавляем новую категорию

Научим бота отвечать на вопрос:

```text
как тебя зовут?
```

Создадим категорию:

```text
name
```

Добавим несколько примеров.

Важно: одного примера обычно мало. Лучше дать разные варианты одной мысли.

In [ ]:
new_texts = [
    "как тебя зовут",
    "как твое имя",
    "кто ты",
    "назови свое имя",
    "ты кто",
    "у тебя есть имя"
]

new_intents = [
    "name",
    "name",
    "name",
    "name",
    "name",
    "name"
]

data["text"].extend(new_texts)
data["intent"].extend(new_intents)

print("Теперь примеров:", len(data["text"]))

Добавим ответ:

In [ ]:
responses["name"] = "Я учебный ИИ-бот, которого мы создали на Python! 🤖"

# Часть 15. Переобучаем модель

После изменения Dataset недостаточно просто вызвать старую модель.

Нужно снова:

1. построить TF-IDF;
2. преобразовать категории;
3. обучить нейросеть.

Это называется **переобучить модель на новых данных**.

In [ ]:
vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(data["text"])

encoder = LabelEncoder()
y = encoder.fit_transform(data["intent"])

model = MLPClassifier(
    hidden_layer_sizes=(12,),
    solver="lbfgs",
    max_iter=3000,
    random_state=42
)

model.fit(X, y)

print("Модель переобучена!")

In [ ]:
print(bot_answer("как тебя зовут"))
print(bot_answer("ты кто"))
print(bot_answer("назови свое имя"))

## Что сейчас произошло?

Мы нашли проблему:

```text
бот не знал своё имя
```

Добавили новые примеры:

```text
как тебя зовут
кто ты
как твое имя
...
```

После этого переобучили модель.

Получился настоящий цикл улучшения ИИ:

```text
НАШЛИ ОШИБКУ
      ↓
ДОБАВИЛИ ДАННЫЕ
      ↓
ПЕРЕОБУЧИЛИ
      ↓
ПРОВЕРИЛИ
      ↓
СТАЛО ЛУЧШЕ
```

# Часть 16. Самостоятельная работа

Теперь добавьте **одну собственную категорию**.

Выберите что-нибудь одно:

```text
weather     — вопросы о погоде
creator     — кто тебя создал
game        — вопросы про игры
age         — сколько тебе лет
mood        — как дела
help        — что ты умеешь
```

## Требования

Добавьте минимум:

- **5 разных фраз** в Dataset;
- одну новую категорию;
- один ответ в `responses`;
- переобучите модель;
- проверьте минимум 3 новых сообщения.

In [ ]:
# ПРИМЕР — измените его на свою категорию

my_texts = [
    "что ты умеешь",
    "что ты можешь",
    "чем ты можешь помочь",
    "расскажи что умеешь",
    "какие у тебя возможности"
]

my_intents = [
    "help",
    "help",
    "help",
    "help",
    "help"
]

data["text"].extend(my_texts)
data["intent"].extend(my_intents)

responses["help"] = "Я умею распознавать некоторые фразы и отвечать на них с помощью машинного обучения! 🤖"

In [ ]:
# Переобучение после добавления своей категории

vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(data["text"])

encoder = LabelEncoder()
y = encoder.fit_transform(data["intent"])

model = MLPClassifier(
    hidden_layer_sizes=(12,),
    solver="lbfgs",
    max_iter=3000,
    random_state=42
)

model.fit(X, y)

print("Готово!")

In [ ]:
# Проверка

print(bot_answer("что ты умеешь"))
print(bot_answer("какие у тебя возможности"))
print(bot_answer("чем можешь помочь"))

# ⭐ Бонус. Показываем категорию и уверенность

Сделаем специальную функцию для исследования работы ИИ.

Она покажет:

- что написал пользователь;
- какую категорию выбрала модель;
- насколько она уверена.

In [ ]:
def debug_bot(message):
    message_vector = vectorizer.transform([message])

    probabilities = model.predict_proba(message_vector)[0]

    best_index = probabilities.argmax()
    confidence = probabilities[best_index]

    intent = encoder.classes_[best_index]

    print("Сообщение:", message)
    print("Категория:", intent)
    print(f"Уверенность: {confidence * 100:.1f}%")

In [ ]:
debug_bot("привет бот")
print()

debug_bot("расскажи про python")
print()

debug_bot("у слона большие уши")

# Итоги занятия

Сегодня мы прошли полный путь создания простого текстового ИИ:

```text
1. СОЗДАЛИ DATASET
        ↓
2. ПРЕВРАТИЛИ ТЕКСТ В ЧИСЛА
   TfidfVectorizer
        ↓
3. ПРЕВРАТИЛИ КАТЕГОРИИ В ЧИСЛА
   LabelEncoder
        ↓
4. СОЗДАЛИ НЕЙРОСЕТЬ
   MLPClassifier
        ↓
5. ОБУЧИЛИ
   fit()
        ↓
6. СДЕЛАЛИ ПРЕДСКАЗАНИЕ
   predict()
        ↓
7. ПОСМОТРЕЛИ УВЕРЕННОСТЬ
   predict_proba()
        ↓
8. СОЗДАЛИ КОНСОЛЬНОГО БОТА
```

## Главные команды

```python
TfidfVectorizer()
```

превращает текст в числа.

```python
LabelEncoder()
```

превращает названия категорий в числа.

```python
model.fit(X, y)
```

обучает модель.

```python
model.predict(...)
```

выбирает категорию.

```python
model.predict_proba(...)
```

показывает уверенность модели.

---

## Главная мысль

> **ИИ не понимает всё автоматически. Его нужно обучать на хороших и разнообразных примерах.**

Чем лучше Dataset — тем лучше обычно работает наш бот.

# Дополнительное задание, если осталось время

Попробуйте улучшить бота:

1. добавьте ещё **2 категории**;
2. для каждой категории добавьте не менее **5 фраз**;
3. специально используйте разные способы задать один вопрос;
4. протестируйте бота минимум на **10 новых сообщениях**;
5. найдите сообщение, которое он понял неправильно;
6. добавьте подходящий пример в Dataset;
7. переобучите модель и проверьте результат снова.